# 04 — Pixel-Based Classification: GEE `MULTIPROBABILITY` RF & Local Python ML

This unified notebook covers both **Pixel-Based Classification** paths in `LC_Framework`:

1. **Default Cloud Path — GEE `MULTIPROBABILITY` Random Forest (`lc_framework.classification.gee_rf`)**:
   Consumes the RFECV-selected feature subset ([Notebook 03](03_feature_selection_rfecv.ipynb)), trains an Earth Engine `smileRandomForest(...).setOutputMode('MULTIPROBABILITY')` classifier, emits canonical `prob_<id>` bands ordered strictly by `Class Schema`, and schedules EECU-optimized `uint16` `[0, 10000]` tiled exports paired with validated `*.manifest.json` sidecars.
2. **Optional Local / Colab ML Path — `lc_framework.classification.local_ml` (`[ml]`)**:
   Supports `RandomForest`, `ExtraTrees`, `LightGBM`, and `XGBoost` with stratified/group cross-validation, hyperparameter tuning (`RandomizedSearchCV` / `GridSearchCV`), and memory-safe tiled Cloud-Optimized GeoTIFF probability inference.

Both paths produce interchangeable **`Class Probability Stack`** artifacts ready for `Object-Refined Pixel Probabilities` (`Classification Assembly`, [Notebook 06](06_hybrid_classification_assembly.ipynb)) and Olofsson + log-loss evaluation ([Notebook 07](07_accuracy_and_trajectory_evaluation.ipynb)).

In [ ]:
# 1. Colab Bootstrap — install [ml] (plus [gee] if running Section 3A)
# Replace <your-org> with your company GitHub organization
# !pip install -q "lc-framework[ml] @ git+https://github.com/<your-org>/LC_Framework.git"
# Optional if running Section 3A (GEE MULTIPROBABILITY RF):
# !pip install -q "lc-framework[gee,ml] @ git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin

from lc_framework.classification import (
    GEERFParams,
    export_probability_tiles,
    predict_gee_probabilities,
    predict_raster_probabilities,
    train_gee_rf,
    train_pixel_classifier_from_csv,
    tune_pixel_classifier,
)
from lc_framework.config import ClassDefinition, ClassSchema, ProjectConfig
from lc_framework.io import read_probability_stack

## 2. Mount Drive & Load `Class Schema` + RFECV Selected Features (Self-Contained Fallback)
Loads `configs/example_project.yaml` and your RFECV JSON/YAML artifact when available, or initializes clean inline defaults so the notebook runs standalone in Google Colab without cloning the repository.

In [ ]:
# Uncomment in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

CONFIG_PATH = Path("../configs/example_project.yaml")
RFECV_CANDIDATES = (
    Path("outputs/rfecv_2025/rfecv_selected_features.json"),
    Path("outputs/rfecv/rfecv_selected_features.json"),
)
RFECV_ARTIFACT_PATH = next((p for p in RFECV_CANDIDATES if p.exists()), None)

if CONFIG_PATH.exists():
    cfg = ProjectConfig.from_yaml(CONFIG_PATH)
    schema = cfg.class_schema
    TARGET_CRS = cfg.crs
    TARGET_SCALE_M = cfg.scale_m
else:
    cfg = None
    TARGET_CRS = "EPSG:32648"
    TARGET_SCALE_M = 20.0
    schema = ClassSchema(
        classes=(
            ClassDefinition(id=1, name="Forest", color="#1b5e20"),
            ClassDefinition(id=2, name="Cropland", color="#fbc02d"),
            ClassDefinition(id=3, name="Waterbody", color="#0288d1"),
        )
    )

SELECTED_FEATURES = (
    RFECV_ARTIFACT_PATH
    if RFECV_ARTIFACT_PATH is not None
    else ["ndvi_dry", "mndwi_dry", "swir1_wet"]
)

print("Class Schema:", [(c.id, c.name) for c in schema.classes])
print(f"CRS={TARGET_CRS}, scale={TARGET_SCALE_M} m, selected_features={SELECTED_FEATURES}")

## 3A. Cloud Path — GEE `MULTIPROBABILITY` Random Forest
Trains `ee.Classifier.smileRandomForest(...).setOutputMode('MULTIPROBABILITY')` on the RFECV feature subset, emits `prob_<id>` bands ordered by `Class Schema`, and prepares EECU-optimized `uint16` `[0, 10000]` tiled exports alongside validated `*.manifest.json` files.

In [ ]:
RUN_GEE_RF = False  # Flip to True after ee.Initialize(project=...) in Colab

if RUN_GEE_RF:
    import ee
    # ee.Authenticate()
    # ee.Initialize(project="your-gcp-project")

    training_fc = ee.FeatureCollection("projects/your-project/assets/training_samples_2025")
    feature_img = ee.Image("projects/your-project/assets/feature_stack_2025")
    AOI_BOUNDS = (100.0, 0.0, 101.0, 1.0)

    gee_model = train_gee_rf(
        training_fc,
        class_schema=schema,
        selected_features=SELECTED_FEATURES,
        params=GEERFParams(number_of_trees=200, min_leaf_population=2, seed=42),
    )
    prob_img = predict_gee_probabilities(gee_model, feature_img)
    export_res = export_probability_tiles(
        prob_img,
        bounds=AOI_BOUNDS,
        export_prefix="projects/your-project/assets/pixel_probs_2025",
        class_schema=schema,
        encoding="uint16",
        epoch="2025",
        tile_size_deg=0.5,
        scale_m=TARGET_SCALE_M,
        crs=TARGET_CRS,
        destination="drive",
        drive_folder="LC_Framework_Exports",
        manifest_dir=Path("outputs/gee_pixel_probs_2025"),
        dry_run=True,  # Set False to submit GEE tasks & write sidecar manifests
    )
    print(f"Planned {len(export_res.plans)} probability tiles (submitted={len(export_res.tasks)})")
else:
    print("GEE RF skipped (RUN_GEE_RF=False). Proceeding to Local Python ML backend.")

## 3B. Local / Colab ML Path (`RandomForest`, `ExtraTrees`, `LightGBM`, `XGBoost`)
Trains or tunes a local classifier directly from the sample-feature CSV (using only the RFECV-selected features) and runs memory-safe tiled COG probability inference on your multi-band feature GeoTIFF.

In [ ]:
SAMPLES_CSV = Path("data/sample_features_2025.csv")
FEATURE_TIF = Path("data/feature_stack_2025.tif")
OUTPUT_COG = Path("outputs/local_pixel_probs_2025.tif")
CLASSIFIER_FAMILY = "random_forest"  # 'random_forest', 'extra_trees', 'lightgbm', or 'xgboost'

model_result = None
out_prob_path = None

if SAMPLES_CSV.exists():
    model_result = train_pixel_classifier_from_csv(
        SAMPLES_CSV,
        class_schema=schema,
        classifier=CLASSIFIER_FAMILY,
        selected_features=SELECTED_FEATURES,
        tune=True,
        n_iter=12,
        cv=5,
        val_fraction=0.2,
        random_state=42,
    )
    print("CV metrics:", model_result.cv_metrics)
    if model_result.evaluation is not None:
        print("Held-out metrics:", model_result.evaluation.metrics)

    if FEATURE_TIF.exists():
        out_prob_path = predict_raster_probabilities(
            FEATURE_TIF,
            model_result.model,
            OUTPUT_COG,
            encoding="uint16",
            epoch="2025",
            block_size=512,
        )
        print("Wrote Class Probability Stack COG ->", out_prob_path)
else:
    print(
        "Real CSV/raster not found on disk — run Section 4 below for a self-contained "
        "synthetic end-to-end tuning + tiled COG inference demo."
    )

## 4. Self-Contained Synthetic Walkthrough (Runs Out of the Box in Colab)
Generates a synthetic sample table + 4-band feature GeoTIFF (with an extra noise band and shuffled band order), tunes a `RandomForest` classifier strictly on the RFECV feature subset, and emits a validated `uint16` `Class Probability Stack` COG via tiled inference.

In [ ]:
if out_prob_path is None:
    demo_dir = Path(tempfile.mkdtemp(prefix="lc_pixel_demo_"))
    rng = np.random.default_rng(17)

    demo_schema = ClassSchema(
        classes=(
            ClassDefinition(id=1, name="Forest"),
            ClassDefinition(id=2, name="Cropland"),
            ClassDefinition(id=3, name="Waterbody"),
        )
    )
    centers = {1: (2.5, 0.0, 0.0), 2: (0.0, 2.5, 0.0), 3: (0.0, 0.0, 2.5)}
    rows = []
    for cid, (c0, c1, c2) in centers.items():
        for _ in range(25):
            rows.append(
                {
                    "class_id": cid,
                    "ndvi_dry": float(c0 + rng.normal(0, 0.3)),
                    "mndwi_dry": float(c1 + rng.normal(0, 0.3)),
                    "swir1_wet": float(c2 + rng.normal(0, 0.3)),
                    "noise_1": float(rng.normal(0, 1.0)),
                }
            )
    demo_df = pd.DataFrame(rows)

    model_result = tune_pixel_classifier(
        demo_df[["ndvi_dry", "mndwi_dry", "swir1_wet", "noise_1"]],
        demo_df["class_id"],
        class_schema=demo_schema,
        classifier="random_forest",
        selected_features=["ndvi_dry", "mndwi_dry", "swir1_wet"],
        param_space={"n_estimators": [20, 40], "max_depth": [3, 5]},
        search_method="grid",
        cv=3,
        val_fraction=0.2,
        n_jobs=1,
        random_state=42,
    )
    print("Demo CV metrics:", model_result.cv_metrics)

    # Write synthetic 16x16 feature raster (bands in shuffled order + noise_1)
    h, w = 16, 16
    cube = np.zeros((4, h, w), dtype=np.float32)
    cube[0] = rng.normal(0, 1.0, (h, w))
    cube[2, :8, :], cube[3, :8, :], cube[1, :8, :] = 2.5, 0.0, 0.0
    cube[2, 8:, :8], cube[3, 8:, :8], cube[1, 8:, :8] = 0.0, 2.5, 0.0
    cube[2, 8:, 8:], cube[3, 8:, 8:], cube[1, 8:, 8:] = 0.0, 0.0, 2.5

    demo_tif = demo_dir / "demo_features.tif"
    with rasterio.open(
        demo_tif,
        "w",
        driver="GTiff",
        dtype="float32",
        count=4,
        height=h,
        width=w,
        crs="EPSG:32648",
        transform=from_origin(500000.0, 1000000.0, 10.0, 10.0),
    ) as dst:
        dst.write(cube)
        for idx, bname in enumerate(("noise_1", "swir1_wet", "ndvi_dry", "mndwi_dry"), start=1):
            dst.set_band_description(idx, bname)

    out_prob_path = predict_raster_probabilities(
        demo_tif,
        model_result.model,
        demo_dir / "demo_pixel_probs.tif",
        encoding="uint16",
        epoch="2025",
        block_size=8,
    )
    print("Demo Class Probability Stack written ->", out_prob_path)

## 5. Verify Emitted `Class Probability Stack` Contract
Reads back the generated COG + sidecar `*.manifest.json` using `read_probability_stack()` to confirm `paradigm="pixel"`, class band descriptions, and sum-to-one invariants.

In [ ]:
if out_prob_path is not None and out_prob_path.exists():
    stack = read_probability_stack(out_prob_path)
    valid_sums = stack.probabilities[:, stack.valid_mask].sum(axis=0)
    print("Paradigm:", stack.manifest.paradigm)
    print("Encoding:", stack.manifest.dtype, "scale_factor:", stack.manifest.scale_factor)
    print("Classes:", stack.manifest.class_schema.class_names)
    print("Probability sum range:", float(valid_sums.min()), "to", float(valid_sums.max()))